# UMUD segment-then-measure (Kaggle GPU) &nbsp;·&nbsp; **version 2026-06-09.5**

> **How to know it's the current code:** cell 1 prints `downloaded pipeline VERSION: 2026-06-09.5` and will *error* if GitHub served a stale script. Each run also prints `##### UMUD pipeline VERSION 2026-06-09.5 #####` with the active flags at the top. If you don't see those, you're on old code — re-run cell 1.

Trains the aponeurosis and fascicle U-Nets, predicts masks on the 309 test images, measures geometry (PCA-fit pennation + length-weighted median, TTA-denoised masks), applies the **per-family tick/ruler scale router** (`scale_ticks.py`) for muscle thickness on ~254/309 images, derives fascicle length from the MT/sin(PA) identity, and writes `submission_segmentation.csv`.

**How to run (no local setup needed):**
1. Open this notebook on Kaggle (New Notebook, then File > Import Notebook, upload this `.ipynb`).
2. Right panel > **Add Input** > add the competition *UMUD Challenge: Muscle Architecture in Ultrasound Data*.
3. Right panel > Settings: **Accelerator = GPU** (T4 or P100) and **Internet = On**.
4. **Run All**. The optional smoke cell (2 epochs) finishes in a few minutes; the full cell (16 epochs here) takes roughly 40-60 minutes.
5. Cell 6 self-checks that calibration ran (~254 scaled). Cell 7 gives download links: the submission, the calibration debug CSV, and the trained weights (`seg_apo.pt`, `seg_fasc.pt`).

**Note:** this competition is a CSV upload. The repo already contains a validated `results/submission_local.csv` (same pipeline on the existing saved weights, CPU). To just see the score without a 40-60 min retrain, upload that CSV directly. Run this notebook when you want fresh weights or a reproducible end-to-end run.

In [ ]:
# Dependencies and the latest pipeline scripts (needs Internet = On)
!pip install -q segmentation-models-pytorch albumentations
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/segment_then_measure.py -O segment_then_measure.py
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/tick_calibration.py -O tick_calibration.py
!wget -q https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture/scale_ticks.py -O scale_ticks.py
import os
for f in ('segment_then_measure.py', 'tick_calibration.py', 'scale_ticks.py'):
    assert os.path.exists(f), f'{f} failed to download - check Internet = On'
# VERSION GATE: confirm we pulled the CURRENT scripts (GitHub raw can cache for a minute or two).
EXPECTED_VERSION = '2026-06-09.5'
got = next((ln.split('"')[1] for ln in open('segment_then_measure.py') if ln.startswith('PIPELINE_VERSION')), 'MISSING')
print(f'>>> downloaded pipeline VERSION: {got}   (expected {EXPECTED_VERSION}) <<<')
assert got == EXPECTED_VERSION, 'STALE script: GitHub is serving an older version. Wait 1-2 min and re-run THIS cell only.'
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
import os
# ===== training config for THIS run (edit, then Run All) =====
# The scale router (muscle-thickness calibration) is INFERENCE-side and always on; it does NOT need
# retraining. These knobs change how the two segmentation U-Nets are TRAINED.
os.environ['UMUD_EPOCHS'] = '16'            # was 12; modest bump (past ~16-20 risks overfitting the sparse masks)
os.environ['UMUD_FASC_POS_WEIGHT'] = '10'   # recall bias on the FASCICLE model: draw more of each dash, targets the FL bottleneck. 0 = the current weights.
# os.environ['UMUD_CLAHE'] = '1'            # contrast-normalize input (retrains BOTH models). Leave off this run; test it separately.
print('TRAIN config -> epochs', os.environ['UMUD_EPOCHS'], '| fascicle pos_weight', os.environ['UMUD_FASC_POS_WEIGHT'])
print('After the run, DOWNLOAD seg_apo.pt + seg_fasc.pt and score them locally before submitting.')

## Optional smoke run (2 epochs, a few minutes)
Proves the data paths, training loop, and submission writing before committing to the full run. Skip it if you just want the full model.

In [ ]:
!UMUD_EPOCHS=2 python segment_then_measure.py

## Full run
Trains both U-Nets for `UMUD_EPOCHS` epochs (set in the config cell above — **16** for this run, with the fascicle recall bias) and overwrites `submission_segmentation.csv`. Also leaves `seg_apo.pt` and `seg_fasc.pt` in the working dir. Roughly 40-60 minutes at 16 epochs. Watch for `[fasc] using pos_weight=10.0 on BCE` and epoch lines running 0 through 15.

In [ ]:
!python segment_then_measure.py

In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission_segmentation.csv')
print(sub.shape)            # expect (309, 4)
print(sub.describe())

# Sanity checks: the scale router should give per-image MT/FL, not flat constants.
assert sub.shape == (309, 4), f'expected 309x4, got {sub.shape}'
assert sub['mt_mm'].std() > 1.0, 'MT looks constant -> calibration did NOT run (scale_ticks import?)'
assert sub['fl_mm'].std() > 1.0, 'FL looks constant -> identity FL / scale did NOT run'
dbg = pd.read_csv('/kaggle/working/calibration_measurement_debug.csv')
n_scaled = int((dbg['calibration_method'] != 'none').sum())
print(f'calibrated (scaled) images: {n_scaled}/309  | methods: {dbg["calibration_method"].value_counts().to_dict()}')
assert n_scaled > 150, f'only {n_scaled} images scaled - expected ~254; check scale_ticks loaded'
print('OK: per-image MT/FL present and ~', n_scaled, 'images scaled.')

In [ ]:
# Click to download. seg_apo.pt / seg_fasc.pt are the trained weights: grab them to inspect the model locally.
import os
from IPython.display import FileLink, display
for f in ('submission_segmentation.csv', 'calibration_measurement_debug.csv', 'seg_apo.pt', 'seg_fasc.pt'):
    if os.path.exists(f):
        display(FileLink(f))